# small\_fable — full pipeline on Colab (GPU)

Runs the whole thing end-to-end on a free Colab T4:

1. **Stage 1 — SFT**: joint planner + executor. Prints a **held-out validation** line every epoch (plan CE, answer CE, and the *plan-vs-no-plan ablation gap*).
2. **Stage 2a — Rollouts**: cache G hot samples per prompt (the offline RL dataset).
3. **Stage 2b — Off-policy GRPO**: trains from the cache. Prints **held-out reward** every inner epoch so you watch RL move.
4. **Compare**: base vs SFT vs SFT+RL on held-out **and** hard out-of-distribution prompts.
5. **Head-to-head**: SFT vs RL on a single hard prompt that appears **nowhere** in the data.

> **Before running:** `Runtime → Change runtime type → T4 GPU`. Then `Runtime → Run all`.
> Each script streams progress live (`python -u`), so you see incremental logs as they train.


## 0 · GPU check

In [ ]:
!nvidia-smi || echo 'NO GPU — set Runtime → Change runtime type → T4 GPU, then re-run.'


## 1 · Clone the repo & install deps
The repo is **public**, so this clones with no auth. Data ships in the repo, so there's nothing else to upload.

In [ ]:
import os
REPO = 'https://github.com/sinha-k-prat/small_fable.git'
if not os.path.isdir('small_fable'):
    !git clone -q $REPO
else:
    !cd small_fable && git pull -q
%cd small_fable
!pip install -q -r requirements.txt
print('\nReady. Files:'); !ls -1


## 2 · Peek at the data
Each row is an `{instruction, plan, answer, checker}` example. The plan is a short symbolic program.

In [ ]:
import json
rows = [json.loads(l) for l in open('dataset/sft_100.jsonl')]
print(f'{len(rows)} examples\n')
for r in rows[:3]:
    print('•', r['instruction'])
    print('  plan  :', ' → '.join(r['plan']))
    print('  answer:', r['answer'], '\n')


## 3 · Stage 1 — Joint SFT
Trains the planner head + executor backbone (LoRA) together. **Watch the `held {...}` line each epoch** — that's the held-out validation set (30 prompts the model never trains on):

- `plan_ce`, `resp_ce` should **fall**.
- `ablation_gap` should be **positive and grow** — proof the plan is load-bearing, not decoration.

_~15–30 min on a T4. The per-epoch held-out generation is the slow part — that's the validation you asked for._

In [ ]:
!python -u train_sft.py \
    --data dataset/sft_100.jsonl --train 70 --held 30 \
    --epochs 6 --lr 2e-5 --bs 4 --max_resp 64 \
    --probe 12 --out joint_ckpt --device cuda


## 4 · Stage 2a — Offline rollouts
Sample `G=8` answers per prompt at a **hot** temperature so the group genuinely differs (that spread is the RL signal). Cached once to `rl_rollouts.jsonl`; training does **no** generation.

Watch `mean_reward` and `zero_var` — too many zero-variance groups means SFT diversity was too low.

_~10–20 min on a T4._

In [ ]:
!python -u rollout_offline.py \
    --sft_ckpt joint_ckpt --data dataset/sft_100.jsonl \
    --train 80 --group 8 --temp 1.5 --top_p 0.98 --max_resp 64 \
    --out rl_rollouts.jsonl --device cuda


## 5 · Stage 2b — Off-policy GRPO
Trains from the cached rollouts. Each inner epoch reuses the same data, corrected by the **clipped importance ratio**. **Watch `held_reward=` every inner epoch** — that's the held-out validation moving under RL.

Sanity lines to look for:
- `logp_mismatch_t0 ≈ 0` — the offline ratio is valid at step 0.
- `>0 trainable backbone tensors` — RL isn't a silent no-op.
- `exec_approx_kl` staying under `kl_stop` — rollouts aren't stale.
- final `|ΔL2| > 0` — the adapter actually moved (RL ≠ SFT).

In [ ]:
!python -u train_grpo_offline.py \
    --rollouts rl_rollouts.jsonl --sft_ckpt joint_ckpt --data dataset/sft_100.jsonl \
    --out rl_ckpt --inner_epochs 3 --lr 1e-4 --clip_eps 0.2 \
    --beta_plan 1.0 --beta_ce 0.1 --held 16 --device cuda


## 6 · Compare — base vs SFT vs SFT+RL
Three-way eval on held-out **and** built-in out-of-distribution prompts. `--sample` (temp 0.7) is required to see small RL effects; greedy decoding hides them.

In [ ]:
!python -u compare.py \
    --sft_ckpt joint_ckpt --rl_ckpt rl_ckpt \
    --data dataset/sft_100.jsonl --train 80 --held 20 \
    --sample --device cuda


## 7 · Head-to-head on a brand-new hard prompt
A multi-step reasoning prompt that appears **nowhere** in the training data. We load both checkpoints and print each model's **plan** and **answer** side by side, so you can see SFT vs RL behavior directly.

_Edit `HARD_PROMPT` to try your own._

In [ ]:
import torch
from model_joint import JointModel, decode_plan

HARD_PROMPT = (
    'A snail is at the bottom of a 12-meter well. Each day it climbs 4 meters, '
    'but each night it slides back 3 meters. On which day does it first reach the top?'
)

def run(ckpt, label, n=3, temp=0.7):
    m = JointModel.from_checkpoint('Qwen/Qwen2.5-1.5B-Instruct', ckpt, device='cuda')
    m.eval()
    print(f'\n{"="*70}\n{label}  ({ckpt})\n{"="*70}')
    with torch.no_grad():
        for k in range(n):
            p_ids, p_attn = m.batch_prompts([HARD_PROMPT])
            plan = m.sample_plan(p_ids, p_attn, temp=temp, sample=True)
            gen = m.generate_answer(p_ids, p_attn, plan, temp=temp, sample=True, max_new_tokens=64)
            ans = m.tok.decode(gen[0], skip_special_tokens=True)
            print(f'  sample {k+1}')
            print(f'    plan  : {" → ".join(decode_plan(plan[0]))}')
            print(f'    answer: {ans.strip()}')
    del m; torch.cuda.empty_cache()

print('HARD PROMPT (not in data):\n ', HARD_PROMPT)
run('joint_ckpt', 'SFT only')
run('rl_ckpt',    'SFT + GRPO')


## 8 · (optional) Save checkpoints to Google Drive
Colab is ephemeral — checkpoints vanish when the runtime recycles. Run this to copy them to Drive.

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/small_fable_ckpts
# !cp -r joint_ckpt rl_ckpt rl_rollouts.jsonl /content/drive/MyDrive/small_fable_ckpts/
# print('saved to Drive')
